In [0]:
dbutils.widgets.text("Landing_Timestamp", "", "Landing timstamps")
dbutils.widgets.text("Table_Name", "", "Table name")
dbutils.widgets.text("Load_Type", "", "Load type")
dbutils.widgets.text("Merge_Keys", "", "Merge keys")

landing_timstamps = dbutils.widgets.get("Landing_Timestamp")
table_name = dbutils.widgets.get("Table_Name")
load_type = dbutils.widgets.get("Load_Type")
merge_keys = dbutils.widgets.get("Merge_Keys")

print("=" * 80)
print("[STEP 1] Job parameters received")
print("=" * 80)
print(f"  Landing_Timestamp : '{landing_timstamps}'")
print(f"  Table_Name        : '{table_name}'")
print(f"  Load_Type         : '{load_type}'")
print(f"  Merge_Keys        : '{merge_keys}'")
if not table_name:
    print("  WARNING: Table_Name is empty")
if not landing_timstamps:
    print("  WARNING: Landing_Timestamp is empty - the bronze filter will likely return 0 rows")

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timstamps}'"

print("=" * 80)
print("[STEP 2] Reading batch from bronze")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Equivalent SQL query:")
print(f"    SELECT *\n    FROM {source_table}\n    WHERE {filter_condition}")

df = spark.read.table(source_table).filter(filter_condition)

source_count = df.count()
print(f"  Rows read from bronze : {source_count}")
print(f"  Columns ({len(df.columns)}) : {df.columns}")
if source_count == 0:
    print("  WARNING: No rows matched the landing timestamp filter")


In [0]:
from pyspark.sql.functions import current_timestamp

print("=" * 80)
print("[STEP 3] Adding audit columns")
print("=" * 80)
print("  Adding: ingestion_timestamp = current_timestamp()")
print("  Adding: update_timestamp    = current_timestamp()")

updated_df = df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

print("  Equivalent SQL query:")
print(f"    SELECT *, current_timestamp() AS ingestion_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")
print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, current_timestamp

target_table = f"adb_caresync.silver.{table_name}"
load_type_norm = load_type.strip().upper()
source_query = f"(SELECT *, current_timestamp() AS ingestion_timestamp, current_timestamp() AS update_timestamp FROM {source_table} WHERE {filter_condition})"

print("=" * 80)
print("[STEP 4] Writing to silver")
print("=" * 80)
print(f"  Target table           : {target_table}")
print(f"  Load type (normalized) : {load_type_norm}")
print(f"  Target exists          : {spark.catalog.tableExists(target_table)}")

if load_type_norm == "FULL":
    # Replace the target with the current batch
    print("  Mode: FULL -> overwriting target with current batch")
    print("  Equivalent SQL query:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS\n    {source_query}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL overwrite completed")

elif load_type_norm == "APPEND":
    # Add the current batch to the target as-is
    print("  Mode: APPEND -> appending current batch to target")
    print("  Equivalent SQL query:")
    print(f"    INSERT INTO {target_table}\n    {source_query}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND completed")

elif load_type_norm == "MERGE":
    print("  Mode: MERGE -> upserting current batch on merge keys")
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"  Parsed merge keys: {keys}")
    if not keys:
        raise ValueError("Merge_Keys must be provided (comma-separated) when Load_Type is MERGE")

    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")
    print("  All merge keys found in source columns")

    target_exists = spark.catalog.tableExists(target_table)
    target_is_empty = (not target_exists) or spark.read.table(target_table).limit(1).count() == 0

    if target_is_empty:
        # Silver has no data yet -> FULL initial load from ALL bronze batches (latest version per key)
        print("  Target is missing or empty -> FULL initial load from ALL bronze batches")
        full_df = (spark.read.table(source_table)
                   .withColumn("ingestion_timestamp", current_timestamp())
                   .withColumn("update_timestamp", current_timestamp()))
        latest_first = Window.partitionBy(*keys).orderBy(col("landing_timestamp").desc())
        source_df = full_df.withColumn("_rn", row_number().over(latest_first)).filter(col("_rn") == 1).drop("_rn")
        before_dedup = full_df.count()
        after_dedup = source_df.count()
        print(f"  Deduplication on {keys} (latest landing_timestamp wins): {before_dedup} rows -> {after_dedup} rows")
        print("  Equivalent SQL query:")
        print(f"    CREATE OR REPLACE TABLE {target_table} AS\n    SELECT *, current_timestamp() AS ingestion_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    QUALIFY ROW_NUMBER() OVER (PARTITION BY {', '.join(keys)} ORDER BY landing_timestamp DESC) = 1")
        source_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
        updated_df = source_df  # returned record count reflects the full load
        print("  FULL initial load completed")
    else:
        # Silver already has data -> INCREMENTAL merge of the current batch only
        print("  Target has data -> INCREMENTAL merge of the current batch")
        # Avoid multiple source rows matching the same target row
        source_df = updated_df.dropDuplicates(keys)
        before_dedup = updated_df.count()
        after_dedup = source_df.count()
        print(f"  Deduplication on {keys}: {before_dedup} rows -> {after_dedup} rows ({before_dedup - after_dedup} duplicates removed)")

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update keep the original ingestion_timestamp, refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "ingestion_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        print(f"  Merge condition : {merge_condition}")
        print(f"  Update columns  : {list(update_set.keys())} (ingestion_timestamp preserved)")
        print(f"  Insert columns  : {list(insert_set.keys())}")

        update_clause = ",\n        ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols = ", ".join(insert_set.keys())
        insert_vals = ", ".join(insert_set.values())
        merge_sql = (
            f"    MERGE INTO {target_table} AS t\n"
            f"    USING {source_query} AS s\n"
            f"    ON {merge_condition}\n"
            f"    WHEN MATCHED THEN UPDATE SET\n        {update_clause}\n"
            f"    WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
            f"    VALUES ({insert_vals})"
        )
        print("  Equivalent SQL query (source is deduplicated on merge keys before merge):")
        print(merge_sql)

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute())
        print("  MERGE completed")

else:
    raise ValueError(f"Unsupported Load_Type '{load_type}'. Expected FULL, APPEND or MERGE")

# Report what Delta actually did in the last operation
last_op = DeltaTable.forName(spark, target_table).history(1).select("version", "operation", "operationMetrics").collect()[0]
print("  Delta last operation:")
print(f"    Version   : {last_op['version']}")
print(f"    Operation : {last_op['operation']}")
for metric, value in sorted((last_op['operationMetrics'] or {}).items()):
    if metric.startswith("num"):
        print(f"    {metric:<30}: {value}")
print(f"  Total rows now in {target_table}: {spark.read.table(target_table).count()}")

In [0]:
record_count = updated_df.count()

print("=" * 80)
print("[STEP 5] Finished")
print("=" * 80)
print(f"  Table           : {table_name}")
print(f"  Load type       : {load_type_norm}")
print(f"  Records processed (returned to caller): {record_count}")

dbutils.notebook.exit(record_count)